# Aufgabe 6c: Diagonale Prototypfilter mit Top-1%-Pooling

Wir übernehmen das Modell aus 06b und ändern ausschließlich das Pooling:

`d²_k(x) = mean_j(a_kj * (x_j − c_kj)²)` → `ReLU(rho_k − d²_k(x))` → **Mittelwert der höchsten 1 % Antworten** → bestehender linearer Output.

Zentren, positive normierte Markergewichte, lernbare Radien, Initialisierung und Output-L2 bleiben gegenüber 06b unverändert. Die Top-k-Zahl folgt exakt 04c: `max(1, int(0.01 * n_cells))`. Falls weniger Zellen positiv antworten, gehen auch Nullen in diese feste Top-k-Mittelung ein.

Die Hypothese lautet, dass das globale Mittel aus 06b seltene Antworten zu stark abschwächt. Der neue Versuch prüft den Einfluss des Poolings bei sonst gleichem Verfahren. Gegenüber der linearen Baseline bleibt die Filterantwort verändert und greift damit den Hinweis aus Aufgabe 6 auf. Eine Verbesserung ist nicht garantiert.

Zuerst 06a und 06b ausführen. Deren Artefakte werden ausschließlich gelesen; sämtliche neuen Ergebnisse heißen `task6_mahalanobis_top1_*`. Die bestehenden Notebooks bleiben unverändert.

In [1]:
from pathlib import Path
import copy
import csv
import hashlib
import json
import os
import sys
import time
import warnings
from importlib.metadata import version

import flowkit as fk
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from IPython.display import display

BONUS_SPLIT_IDS = [0, 1, 2]
SPLIT_IDS = BONUS_SPLIT_IDS
GATE, RUN_MODE, GATE_SUFFIX = "gated_alive", "full", "_alive"
COFACTOR, TOP_FRACTION = 5.0, 0.01
LEARNING_RATE, L2_COEFFICIENT = 0.01, 1e-4
TRAINING_CELLS_PER_INPUT, TRAINING_INPUTS_PER_DONOR = 3000, 200
PREDICTION_CELLS_PER_INPUT, PREDICTION_INPUTS_PER_DONOR = 20000, 5
SCALER_CELLS_PER_DONOR = 20000
FILTER_COUNTS, BATCH_SIZE = [3, 4, 5], 128
MAX_EPOCHS, EARLY_STOPPING_PATIENCE = 100, 5
EVALUATION_CELLS_PER_DONOR, EVALUATION_SEED = 20000, 63000
EXPECTED_EVENT_COUNTS = {"gated_alive": 3438750}

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "NK_cell_dataset").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT / "NK_cell_dataset/NK_cell_dataset"
FCS_DIR = DATA_ROOT / "NK_cell_dataset" / GATE
LABELS_PATH, MARKERS_PATH = DATA_ROOT / "NK_fcs_samples_with_labels.csv", DATA_ROOT / "NK_markers.csv"
TABLES = PROJECT_ROOT / "results/tables"
SPLITS_PATH = TABLES / "task4_donor_splits.csv"
sys.path.insert(0, str(PROJECT_ROOT))
from src.task4_artifacts import make_run_config, file_digest, validate_prediction_splits, validate_parameter_table
from src.task5_interpretation import SavedCellCNN, restore_scaler

torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Gerät: {DEVICE}; PyTorch: {torch.__version__}; Bonus-Splits: {BONUS_SPLIT_IDS}")


Gerät: cuda; PyTorch: 2.14.0+cu130; Bonus-Splits: [0, 1, 2]


In [2]:
# Zweck: Alle Spenderdaten transformiert, aber weiterhin spenderweise getrennt laden.
with MARKERS_PATH.open(newline="", encoding="utf-8-sig") as stream:
    markers = next(csv.reader(stream))

label_table = pd.read_csv(LABELS_PATH)
label_table["donor_id"] = label_table["fcs_filename"].str.replace(
    r"_NK\.fcs$", "", regex=True
)
label_table["label"] = label_table["label"].astype(int)
fcs_paths = sorted(FCS_DIR.glob("*.fcs"))
fcs_table = pd.DataFrame(
    {
        "donor_id": [path.stem.removesuffix(GATE_SUFFIX) for path in fcs_paths],
        "fcs_path": fcs_paths,
    }
)
sample_table = (
    label_table[["donor_id", "label"]]
    .merge(fcs_table, on="donor_id", validate="one_to_one")
    .sort_values("donor_id")
    .reset_index(drop=True)
)
donor_splits = pd.read_csv(SPLITS_PATH)

assert len(markers) == 37
assert len(fcs_paths) == 20
assert len(sample_table) == 20
assert set(SPLIT_IDS).issubset(set(donor_splits["split_id"]))
split_labels = donor_splits[["donor_id", "label"]].drop_duplicates()
assert split_labels.merge(
    sample_table[["donor_id", "label"]],
    on=["donor_id", "label"],
    validate="one_to_one",
).shape[0] == len(sample_table)

def load_transformed_fcs(path: Path) -> np.ndarray:
    """Eine FCS-Datei read-only als ArcSinh-transformierte Markermatrix laden."""
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore", message=r"FCS file .* reported incorrect data offset.*"
        )
        sample = fk.Sample(str(path), ignore_offset_error=True)
    frame = sample.as_dataframe(source="raw")
    short_names = pd.Index(sample.pns_labels, name="marker")
    if not short_names.is_unique:
        raise ValueError(f"Doppelte FCS-Kurznamen in {path.name}.")
    frame.columns = short_names
    missing = [marker for marker in markers if marker not in frame.columns]
    if missing:
        raise ValueError(f"Fehlende Marker in {path.name}: {missing}")
    values = frame.loc[:, markers].to_numpy(dtype=np.float32, copy=True)
    values = np.arcsinh(values / COFACTOR).astype(np.float32, copy=False)
    if not np.isfinite(values).all():
        raise ValueError(f"Nicht-endliche Werte in {path.name}.")
    return values

# Die getrennte Ablage verhindert ein versehentliches Aufteilen eines Spenders.
data_by_donor = {
    row.donor_id: load_transformed_fcs(row.fcs_path)
    for row in sample_table.itertuples(index=False)
}
label_by_donor = sample_table.set_index("donor_id")["label"].to_dict()
assert sum(map(len, data_by_donor.values())) == EXPECTED_EVENT_COUNTS[GATE]

display(
    pd.DataFrame(
        {
            "donor_id": list(data_by_donor),
            "label": [label_by_donor[x] for x in data_by_donor],
            "cells": [len(data_by_donor[x]) for x in data_by_donor],
        }
    )
)


,donor_id,label,cells
0,a_001,1,82324
1,a_002,1,108267
2,a_003,0,97529
3,a_004,0,140687
4,a_005,1,155335
5,a_006,0,90075
6,a_007,1,104805
7,a_009,0,216632
8,a_010,0,122209
9,a_011,0,258461


In [3]:
# Referenzcode und Eingabedaten prüfen, ohne 04c auszuführen oder Artefakte zu schreiben.
baseline_path = TABLES / "task4_cellcnn_predictions_gated_alive_full.csv"
baseline_config_path = baseline_path.with_suffix(".config.json")
baseline_config = json.loads(baseline_config_path.read_text())
expected_parameters = {
    "method": "cellcnn", "gate": GATE, "run_mode": RUN_MODE,
    "cofactor": COFACTOR, "learning_rate": LEARNING_RATE,
    "l2_coefficient": L2_COEFFICIENT, "top_fraction": TOP_FRACTION,
    "training_cells_per_input": TRAINING_CELLS_PER_INPUT,
    "training_inputs_per_donor": TRAINING_INPUTS_PER_DONOR,
    "prediction_cells_per_input": PREDICTION_CELLS_PER_INPUT,
    "prediction_inputs_per_donor": PREDICTION_INPUTS_PER_DONOR,
    "scaler_cells_per_donor": SCALER_CELLS_PER_DONOR,
    "filter_counts": FILTER_COUNTS, "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS, "patience": EARLY_STOPPING_PATIENCE,
    "threshold": 0.5, "implementation_version": "pytorch_materialized_v1",
    "model_format": "complete_filter_parameters_v1",
}
expected_config = make_run_config(
    baseline_config["parameters"] | expected_parameters,
    [SPLITS_PATH, LABELS_PATH, MARKERS_PATH, *fcs_paths],
    PROJECT_ROOT / "notebooks/04c_cellcnn.ipynb", [4, 6, 8, 10],
)
if baseline_config != expected_config:
    raise ValueError("Baseline-Konfiguration, Referenzcode oder Eingangsdaten passen nicht.")

baseline_predictions = pd.read_csv(baseline_path, float_precision="round_trip")
baseline_filters_path = TABLES / "task4_cellcnn_filters_gated_alive_full.csv"
baseline_filters = pd.read_csv(baseline_filters_path, float_precision="round_trip")
baseline_selection = pd.read_csv(TABLES / "task4_cellcnn_selection_gated_alive_full.csv")
baseline_predictions = baseline_predictions.loc[baseline_predictions.split_id.isin(BONUS_SPLIT_IDS)].copy()
baseline_filters = baseline_filters.loc[baseline_filters.split_id.isin(BONUS_SPLIT_IDS)].copy()
validate_prediction_splits(baseline_predictions, donor_splits)
validate_parameter_table(baseline_filters, baseline_predictions, markers, ["split_id", "filter_id"],
    ["filter_weight", "filter_bias", "output_weight_0", "output_weight_1",
     "output_bias_0", "output_bias_1", "output_weight_contrast", "scaler_mean", "scaler_scale"])
assert set(baseline_predictions.split_id) == set(BONUS_SPLIT_IDS)
for split_id in BONUS_SPLIT_IDS:
    split = donor_splits.loc[donor_splits.split_id.eq(split_id)]
    assert len(split) == 20 and split.donor_id.nunique() == 20
    assert split.outer_partition.eq("test").sum() == 6
    assert split.outer_partition.eq("train").sum() == 14
    assert set(split.loc[split.outer_partition.eq("train"), "inner_fold"]) == {0, 1, 2}
    selection = baseline_selection.loc[baseline_selection.split_id.eq(split_id)]
    assert len(selection) == 9
    assert set(zip(selection.inner_fold, selection.filter_count)) == {(f, k) for f in range(3) for k in FILTER_COUNTS}
    best = selection.sort_values(
        ["validation_accuracy", "validation_roc_auc", "best_validation_loss", "filter_count", "inner_fold"],
        ascending=[False, False, True, True, True]).iloc[0]
    assert selection.selected.sum() == 1 and bool(best.selected)
    filters = baseline_filters.loc[baseline_filters.split_id.eq(split_id)]
    pred = baseline_predictions.loc[baseline_predictions.split_id.eq(split_id)]
    assert filters.inner_fold.eq(best.inner_fold).all() and pred.selected_inner_fold.eq(best.inner_fold).all()
    assert filters.filter_id.nunique() == best.filter_count and pred.filter_count.eq(best.filter_count).all()
    assert np.allclose(filters.output_weight_contrast, filters.output_weight_1 - filters.output_weight_0)
    assert filters.scaler_scale.gt(0).all()
    for frame in [filters, pred]:
        assert frame.gate.eq(GATE).all() and frame.run_mode.eq(RUN_MODE).all()
        assert frame.top_fraction.eq(TOP_FRACTION).all()
    assert pred.score.between(0, 1).all()

# Identische Originalereignisse je Spender, unabhängig von Modell und Split.
evaluation_indices = {
    donor: np.random.default_rng(EVALUATION_SEED + i).choice(
        len(data_by_donor[donor]), min(EVALUATION_CELLS_PER_DONOR, len(data_by_donor[donor])), replace=False)
    for i, donor in enumerate(sorted(data_by_donor))
}
evaluation_config = {
    "baseline_config": baseline_config,
    "baseline_predictions_sha256": file_digest(baseline_path),
    "baseline_filters_sha256": file_digest(baseline_filters_path),
    "split_ids": BONUS_SPLIT_IDS, "evaluation_seed": EVALUATION_SEED,
    "evaluation_cells_per_donor": EVALUATION_CELLS_PER_DONOR,
    "selection": "largest_positive_output_contrast", "baseline_membership": "half_inner_training_maximum",
    "event_indices_sha256": hashlib.sha256(b"".join(
        evaluation_indices[d].astype("<i8").tobytes() for d in sorted(evaluation_indices))).hexdigest(),
}
print("Baseline-Artefakte, Referenzcode und spenderweise Splits geprüft.")


Baseline-Artefakte, Referenzcode und spenderweise Splits geprüft.


## Modell und unveränderte Trainingsgrundlage

Die folgenden Sampling-, Skalierungs-, Trainings- und Auswahlfunktionen stammen aus 04c. Angepasst sind Modellkonstruktion, Initialisierung, L2-Umfang und Export. Die neun Kandidaten je Outer-Split (drei innere Folds × drei Filterzahlen) werden wie bisher nach Validierungsgenauigkeit, AUC, Verlust und festen Tie-Breakern ausgewählt. Das ausgewählte innere Modell wird direkt getestet, ohne Refit auf allen äußeren Trainingsspendern.

Prototypen und 1%-Radien werden ausschließlich aus gleich vielen Zellen je innerem Trainingsspender initialisiert (maximal 2.000). Ein eigener NumPy-Seed hält die bisherigen Bag-Sequenzen unverändert. Die anfänglichen Markergewichte sind eins; die Normierung beseitigt deren gemeinsame freie Skalierung. L2 bleibt nur auf den Output-Gewichten. Gegenüber 06b ändert sich nur die Aggregation der Zellantworten. Der Smoke-Test kontrolliert das Top-1%-Pooling, aktive Filter und lernende Parameter.

In [4]:
INITIALIZATION_CELLS_PER_DONOR = 2000
INITIALIZATION_SEED_OFFSET = 300000
EPS = 1e-6
IMPLEMENTATION_VERSION = "diagonal_prototype_relu_top1_v1"


class PrototypeCellCNN(nn.Module):
    """Diagonale Prototypdistanz, ReLU-Radius und Top-1%-Mittelwert."""
    def __init__(self, marker_count, filter_count):
        super().__init__()
        self.centers = nn.Parameter(torch.zeros(filter_count, marker_count))
        self.raw_a = nn.Parameter(torch.zeros(filter_count, marker_count))
        self.raw_rho = nn.Parameter(torch.zeros(filter_count))
        self.output_layer = nn.Linear(filter_count, 2)

    def marker_weights(self):
        positive = nn.functional.softplus(self.raw_a) + EPS
        return positive / positive.mean(dim=-1, keepdim=True)

    def radii(self):
        return nn.functional.softplus(self.raw_rho) + EPS

    def distances(self, values):
        a = self.marker_weights()
        distance = (values.square() @ a.T - 2 * values @ (a * self.centers).T
                    + (a * self.centers.square()).sum(dim=-1)) / values.shape[-1]
        return distance.clamp_min(0)

    def responses(self, values):
        return torch.relu(self.radii() - self.distances(values))

    def forward(self, values):
        responses = self.responses(values)
        top_count = max(1, int(TOP_FRACTION * values.shape[1]))
        pooled = torch.topk(responses, k=top_count, dim=1).values.mean(dim=1)
        return self.output_layer(pooled)


def initialize_prototypes(model, train_ids, scaled_data, seed):
    # Gleiche Zellzahl je Trainingsspender; eigener RNG beeinflusst keine Bags.
    rng = np.random.default_rng(seed + INITIALIZATION_SEED_OFFSET)
    count = min(INITIALIZATION_CELLS_PER_DONOR, min(len(scaled_data[d]) for d in train_ids))
    sample = np.concatenate([scaled_data[d][rng.choice(len(scaled_data[d]), count, replace=False)]
                             for d in sorted(train_ids)])
    selected = rng.choice(len(sample), len(model.centers), replace=False)
    sample_tensor = torch.from_numpy(sample).to(DEVICE)
    with torch.no_grad():
        model.centers.copy_(sample_tensor[selected])
        model.raw_a.zero_()
        radius = torch.quantile(model.distances(sample_tensor), 0.01, dim=0).clamp_min(2 * EPS)
        positive_part = radius - EPS
        model.raw_rho.copy_(positive_part + torch.log(-torch.expm1(-positive_part)))
    return sample_tensor


In [5]:
# Zweck: Spenderbalancierte Multi-Cell-Inputs erzeugen und die CellCNN-Schichten definieren.
def materialize_multicell_inputs(
    donor_ids: list[str],
    scaled_data: dict[str, np.ndarray],
    cells_per_input: int,
    inputs_per_donor: int,
    seed: int,
) -> TensorDataset:
    """Feste zufällige Zellgruppen mit je einem Spenderlabel materialisieren.

    Jeder Spender erzeugt gleich viele Inputs. Ziehen mit Zurücklegen erhöht
    die Zahl der Trainingsbeispiele, ohne Spender als unabhängig zu vervielfachen.
    """
    donor_ids = sorted(donor_ids)
    input_count = len(donor_ids) * inputs_per_donor
    values = np.empty(
        (input_count, cells_per_input, len(markers)), dtype=np.float32
    )
    labels = np.empty(input_count, dtype=np.int64)
    input_index = 0
    for donor_id in donor_ids:
        donor_values = scaled_data[donor_id]
        for _ in range(inputs_per_donor):
            # Ein eigener deterministischer Teilseed macht jeden Input reproduzierbar.
            rng = np.random.default_rng(seed + input_index)
            cell_indices = rng.integers(
                0, len(donor_values), size=cells_per_input
            )
            values[input_index] = donor_values[cell_indices]
            labels[input_index] = label_by_donor[donor_id]
            input_index += 1
    return TensorDataset(
        torch.from_numpy(values).to(DEVICE),
        torch.from_numpy(labels).to(DEVICE),
    )


def fit_balanced_scaler(donor_ids: list[str], cells_per_donor: int, seed: int):
    """Scaler auf gleich vielen Zellen je Trainingsspender fitten."""
    rng = np.random.default_rng(seed)
    scaler_parts = []
    for donor_id in sorted(donor_ids):
        values = data_by_donor[donor_id]
        if len(values) < cells_per_donor:
            raise ValueError(f"Zu wenige Zellen für Skalierung: {donor_id}")
        indices = rng.choice(len(values), size=cells_per_donor, replace=False)
        scaler_parts.append(values[indices])
    scaler = StandardScaler().fit(np.concatenate(scaler_parts))
    return scaler


def scale_donors(donor_ids: list[str], scaler: StandardScaler):
    """Bereits gefittete Trainingsparameter unverändert auf Spender anwenden."""
    return {
        donor_id: scaler.transform(data_by_donor[donor_id]).astype(np.float32, copy=False)
        for donor_id in donor_ids
    }


In [6]:
# Zweck: Modellkandidaten mit L2-Regularisierung und validierungsbasiertem Stopp trainieren.
def regularized_loss(model: PrototypeCellCNN, logits: torch.Tensor, labels: torch.Tensor):
    """Kreuzentropie plus L2-Strafe der lernbaren Gewichtsmatrizen berechnen."""
    cross_entropy = nn.functional.cross_entropy(logits, labels)
    weight_penalty = (
        model.output_layer.weight.square().sum()
    )
    return cross_entropy + L2_COEFFICIENT * weight_penalty


def evaluate_loader(model: PrototypeCellCNN, loader: DataLoader) -> float:
    """Mittleren regularisierten Verlust ohne Gradienten berechnen."""
    model.eval()
    weighted_loss_sum = 0.0
    example_count = 0
    with torch.no_grad():
        for values, labels in loader:
            logits = model(values.to(DEVICE))
            batch_loss = float(regularized_loss(model, logits, labels.to(DEVICE)))
            weighted_loss_sum += batch_loss * len(labels)
            example_count += len(labels)
    return weighted_loss_sum / example_count


def train_candidate(
    train_ids: list[str],
    validation_ids: list[str],
    filter_count: int,
    seed: int,
) -> tuple[PrototypeCellCNN, StandardScaler, dict[str, object]]:
    """Einen Kandidaten ausschließlich auf innerem Training und Validierung fitten.

    Zurückgegeben werden der beste Modellzustand, sein Trainings-Scaler und
    die Lernhistorie; äußere Testspender werden hier nie verwendet.
    """
    torch.manual_seed(seed)
    scaler = fit_balanced_scaler(train_ids, SCALER_CELLS_PER_DONOR, seed)
    scaled_data = scale_donors(train_ids + validation_ids, scaler)
    train_dataset = materialize_multicell_inputs(
        train_ids, scaled_data, TRAINING_CELLS_PER_INPUT,
        TRAINING_INPUTS_PER_DONOR, seed + 1_000,
    )
    validation_dataset = materialize_multicell_inputs(
        validation_ids, scaled_data, TRAINING_CELLS_PER_INPUT,
        TRAINING_INPUTS_PER_DONOR, seed + 2_000,
    )
    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=0, generator=generator,
    )
    validation_loader = DataLoader(
        validation_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
    )

    model = PrototypeCellCNN(len(markers), filter_count).to(DEVICE)
    initialize_prototypes(model, train_ids, scaled_data, seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    best_state = copy.deepcopy(model.state_dict())
    best_validation_loss = np.inf
    epochs_without_improvement = 0
    history = []

    for epoch in range(MAX_EPOCHS):
        model.train()
        train_losses = []
        for values, labels in train_loader:
            optimizer.zero_grad()
            logits = model(values.to(DEVICE))
            loss = regularized_loss(model, logits, labels.to(DEVICE))
            loss.backward()
            optimizer.step()
            train_losses.append(float(loss.detach()))

        validation_loss = evaluate_loader(model, validation_loader)
        history.append(
            {
                "epoch": epoch + 1,
                "train_loss": float(np.mean(train_losses)),
                "validation_loss": validation_loss,
            }
        )
        # Nur echte Verbesserungen ersetzen den gesicherten besten Zustand.
        if validation_loss < best_validation_loss - 1e-6:
            best_validation_loss = validation_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
                break

    model.load_state_dict(best_state)
    training_info = {
        "epochs_run": len(history),
        "best_validation_loss": best_validation_loss,
        "history": history,
    }
    return model, scaler, training_info


In [7]:
# Zweck: Kandidaten spenderweise bewerten, auswählen und ihre Filter für Aufgabe 5 sichern.
def predict_donor(
    model: PrototypeCellCNN,
    scaler: StandardScaler,
    donor_id: str,
    seed: int,
) -> float:
    """Mehrere zufällige Multi-Cell-Vorhersagen eines Spenders mitteln."""
    values = data_by_donor[donor_id]
    input_cell_count = min(PREDICTION_CELLS_PER_INPUT, len(values))
    rng = np.random.default_rng(seed)
    probabilities = []
    model.eval()
    with torch.no_grad():
        for _ in range(PREDICTION_INPUTS_PER_DONOR):
            indices = rng.choice(len(values), size=input_cell_count, replace=False)
            scaled = scaler.transform(values[indices]).astype(np.float32, copy=False)
            logits = model(torch.from_numpy(scaled).unsqueeze(0).to(DEVICE))
            probability = torch.softmax(logits, dim=1)[0, 1].item()
            probabilities.append(probability)
    return float(np.mean(probabilities))


def train_outer_split(split_id: int):
    """Alle inneren Kandidaten fitten und genau ein Modell extern testen.

    Die Funktion liefert getrennte Tabellen für Testvorhersagen, Auswahlprozess
    und interpretierbare Filterparameter des ausgewählten Netzes.
    """
    split = donor_splits.loc[donor_splits["split_id"] == split_id]
    split_seed = int(split["split_seed"].iloc[0])
    test_ids = split.loc[split["outer_partition"] == "test", "donor_id"].tolist()
    candidates = []
    selection_records = []

    for inner_fold in range(3):
        train_ids = split.loc[
            (split["outer_partition"] == "train")
            & (split["inner_fold"] != inner_fold), "donor_id"
        ].tolist()
        validation_ids = split.loc[
            (split["outer_partition"] == "train")
            & (split["inner_fold"] == inner_fold), "donor_id"
        ].tolist()
        assert not set(train_ids) & set(validation_ids)
        assert not set(train_ids + validation_ids) & set(test_ids)

        for filter_count in FILTER_COUNTS:
            candidate_seed = split_seed + 10_000 * inner_fold + 100 * filter_count
            model, scaler, training_info = train_candidate(
                train_ids, validation_ids, filter_count, candidate_seed
            )
            validation_scores = np.array(
                [
                    predict_donor(model, scaler, donor_id, candidate_seed + 50_000 + index)
                    for index, donor_id in enumerate(sorted(validation_ids))
                ]
            )
            validation_true = np.array(
                [label_by_donor[x] for x in sorted(validation_ids)]
            )
            validation_accuracy = float(
                ((validation_scores >= 0.5).astype(int) == validation_true).mean()
            )
            validation_auc = float(roc_auc_score(validation_true, validation_scores))
            record = {
                "split_id": split_id,
                "gate": GATE,
                "run_mode": RUN_MODE,
                "device": DEVICE.type,
                "implementation_version": IMPLEMENTATION_VERSION,
                "inner_fold": inner_fold,
                "filter_count": filter_count,
                "candidate_seed": candidate_seed,
                "validation_accuracy": validation_accuracy,
                "validation_roc_auc": validation_auc,
                "best_validation_loss": training_info["best_validation_loss"],
                "epochs_run": training_info["epochs_run"],
            }
            selection_records.append(record)
            candidates.append({"model": model, "scaler": scaler, **record})
            print(f"  Fold {inner_fold}, Filter {filter_count}: {training_info['epochs_run']} Epochen", flush=True)

    # Sortierschlüssel kodiert die vorab festgelegte Auswahl samt Tie-Breakern.
    candidates.sort(
        key=lambda item: (
            -item["validation_accuracy"],
            -item["validation_roc_auc"],
            item["best_validation_loss"],
            item["filter_count"],
            item["inner_fold"],
        )
    )
    selected = candidates[0]
    for record in selection_records:
        record["selected"] = (
            record["inner_fold"] == selected["inner_fold"]
            and record["filter_count"] == selected["filter_count"]
        )

    prediction_records = []
    for index, donor_id in enumerate(sorted(test_ids)):
        score = predict_donor(
            selected["model"], selected["scaler"], donor_id,
            split_seed + 900_000 + index,
        )
        prediction_records.append(
            {
                "method": "cellcnn_prototype",
                "gate": GATE,
                "run_mode": RUN_MODE,
                "device": DEVICE.type,
                "implementation_version": IMPLEMENTATION_VERSION,
                "split_id": split_id,
                "split_seed": split_seed,
                "donor_id": donor_id,
                "y_true": label_by_donor[donor_id],
                "score": score,
                "decision_threshold": 0.5,
                "y_pred": int(score >= 0.5),
                "filter_count": selected["filter_count"],
                "selected_inner_fold": selected["inner_fold"],
                "training_cells_per_input": TRAINING_CELLS_PER_INPUT,
                "training_inputs_per_donor": TRAINING_INPUTS_PER_DONOR,
                "prediction_cells_per_input": min(
                    PREDICTION_CELLS_PER_INPUT, len(data_by_donor[donor_id])
                ),
                "prediction_inputs_per_donor": PREDICTION_INPUTS_PER_DONOR,
                "pooling": "top_fraction_mean_thresholded_responses",
                "top_fraction": TOP_FRACTION,
            }
        )

    return pd.DataFrame(prediction_records), pd.DataFrame(selection_records), selected


## Kleiner Smoke-Test vor dem Full-Lauf

Zuerst werden Formel, Initialisierung und Gradienten synthetisch geprüft. Danach durchläuft ein kleiner echter Outer-Split alle neun Kandidaten mit zwei Epochen. Dessen Ergebnisse werden weder gespeichert noch mit Full-Ergebnissen vermischt. Sämtliche Full-Einstellungen werden danach wiederhergestellt.

In [8]:
def smoke_test():
    torch.manual_seed(601)
    rng = np.random.default_rng(601)
    values = rng.normal(size=(512, len(markers))).astype(np.float32)
    model = PrototypeCellCNN(len(markers), 3).to(DEVICE)
    sample = initialize_prototypes(model, ["synthetic"], {"synthetic": values}, 601)
    # Ein fremder Datenblock darf die Initialisierung nicht verändern.
    other = PrototypeCellCNN(len(markers), 3).to(DEVICE)
    initialize_prototypes(other, ["synthetic"], {"synthetic": values, "excluded": values * 1000}, 601)
    assert torch.equal(model.centers, other.centers) and torch.equal(model.raw_rho, other.raw_rho)
    direct = ((sample[:, None, :] - model.centers[None, :, :]).square()
              * model.marker_weights()[None, :, :]).mean(dim=-1)
    torch.testing.assert_close(model.distances(sample), direct, atol=1e-6, rtol=1e-5)
    assert all(torch.any(torch.all(sample == c, dim=1)) for c in model.centers)
    assert torch.all(model.marker_weights() > 0) and torch.all(model.radii() > 0)
    torch.testing.assert_close(model.marker_weights().mean(-1), torch.ones(3, device=DEVICE))
    assert torch.all((model.responses(sample) > 0).sum(0) > 0)
    bags = sample.unsqueeze(0).repeat(2, 1, 1)
    # Sortieren als unabhängige Referenz: 512/257 Zellen prüfen Abrunden, 51 den Mindestwert eins.
    for count in [512, 257, 51]:
        subset = bags[:, :count]
        top_count = max(1, int(TOP_FRACTION * count))
        expected = model.responses(subset).sort(dim=1, descending=True).values[:, :top_count].mean(1)
        torch.testing.assert_close(model(subset), model.output_layer(expected))
    assert not torch.allclose(model(bags), model.output_layer(model.responses(bags).mean(1)))
    # Logits bleiben bei einer Permutation der Zellen gleich.
    torch.testing.assert_close(model(bags), model(bags.flip(1)))
    before = {name: p.detach().clone() for name, p in model.named_parameters()}
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    loss = regularized_loss(model, model(bags), torch.ones(2, dtype=torch.long, device=DEVICE))
    loss.backward()
    assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters())
    optimizer.step()
    for name in ["centers", "raw_a", "raw_rho"]:
        assert not torch.equal(before[name], dict(model.named_parameters())[name]), name
    print("Mathematik-, Initialisierungs-, Gradienten- und Lernfähigkeitscheck bestanden.")

    # Ein kompletter kleiner Outer-Split mit echter Auswahl-/Vorhersagelogik; keine Exporte.
    small = {"TRAINING_CELLS_PER_INPUT": 256, "TRAINING_INPUTS_PER_DONOR": 4,
             "PREDICTION_CELLS_PER_INPUT": 1000, "PREDICTION_INPUTS_PER_DONOR": 2,
             "SCALER_CELLS_PER_DONOR": 1000, "BATCH_SIZE": 16, "MAX_EPOCHS": 2}
    original = {key: globals()[key] for key in small}
    try:
        globals().update(small)
        predictions, selection, chosen = train_outer_split(BONUS_SPLIT_IDS[0])
        validate_prediction_splits(predictions, donor_splits)
        assert len(selection) == 9 and selection.selected.sum() == 1
        assert predictions.score.between(0, 1).all()
        print("Isolierter Smoke-Outer-Split bestanden; kein Full-Ergebnis:",
              roc_auc_score(predictions.y_true, predictions.score))
    finally:
        globals().update(original)

smoke_test()


Mathematik-, Initialisierungs-, Gradienten- und Lernfähigkeitscheck bestanden.


  Fold 0, Filter 3: 2 Epochen


  Fold 0, Filter 4: 2 Epochen


  Fold 0, Filter 5: 2 Epochen


  Fold 1, Filter 3: 2 Epochen


...

  Fold 1, Filter 5: 2 Epochen


  Fold 2, Filter 3: 2 Epochen


  Fold 2, Filter 4: 2 Epochen


  Fold 2, Filter 5: 2 Epochen


Isolierter Smoke-Outer-Split bestanden; kein Full-Ergebnis: 0.125


In [9]:
def strongest_positive_filter(model):
    contrasts = (model.output_layer.weight[1] - model.output_layer.weight[0]).detach().cpu().numpy()
    positive = np.flatnonzero(contrasts > 0)
    return int(positive[np.argmax(contrasts[positive])]) if len(positive) else None


def summarize_split(predictions, frequencies, split_id):
    pred = predictions.loc[predictions.split_id.eq(split_id)]
    freq = frequencies.loc[frequencies.split_id.eq(split_id)]
    valid = len(freq) == len(pred) and freq.frequency.notna().all()
    return {
        "split_id": split_id,
        "network_auc": roc_auc_score(pred.y_true, pred.score),
        "frequency_auc": roc_auc_score(freq.y_true, freq.frequency) if valid else np.nan,
        "frequency_effect": (freq.loc[freq.y_true.eq(1), "frequency"].mean()
                             - freq.loc[freq.y_true.eq(0), "frequency"].mean()) if valid else np.nan,
        "phenotype_status": "berechnet" if valid else "kein positiver Output-Kontrast",
    }


## Full-Ausführung und einfache Wiederaufnahme

Standardmäßig werden nur `[0, 1, 2]` ausgeführt. `TASK6_SPLIT_LIMIT=1` begrenzt die Ausführung auf den ersten Split; `TASK6_RUN_TRAINING=0` verlangt vorhandene Checkpoints. Nach jedem fertig trainierten Split werden Modell, Scaler, Predictions und Kandidatenauswahl in einer gemeinsamen Datei gesichert. Abweichende Konfigurationen werden nicht stillschweigend wiederverwendet. Die Baseline-Auswertung und ihre Zellstichprobe müssen zu 06a passen.

Die Phänotyppopulation erfüllt `d² < rho`. Ihr Anteil ist von der Netzwerkantwort zu unterscheiden: Die Antwort gewichtet zusätzlich die Nähe zum Prototypen. Filterauswahl, Häufigkeitsmetriken und fehlende Werte folgen denselben Regeln wie in 06a.

Die harte Phänotyppopulation umfasst weiterhin alle Zellen mit `d² < rho`; sie ist nicht identisch mit der pro Prediction-Bag gepoolten Top-1%-Auswahl. Diese Membership-Regel bleibt bewusst gleich wie in 06b, damit nur das Netzwerk-Pooling geändert wird.

In [10]:
TRAINING_CODE_CELLS = [1, 2, 5, 6, 7, 8]
# Standard: drei Splits. TASK6_SPLIT_LIMIT=1 erlaubt zunächst nur den ersten Lauf.
split_limit = int(os.environ.get("TASK6_SPLIT_LIMIT", "0"))
active_split_ids = BONUS_SPLIT_IDS[:split_limit] if split_limit > 0 else BONUS_SPLIT_IDS
run_training = os.environ.get("TASK6_RUN_TRAINING", "1") == "1"
baseline_export_config = json.loads((TABLES / "task6_baseline_evaluation.config.json").read_text())
expected_export = evaluation_config | {
    "metrics_sha256": file_digest(TABLES / "task6_baseline_metrics.csv"),
    "frequencies_sha256": file_digest(TABLES / "task6_baseline_frequencies.csv"),
}
if baseline_export_config != expected_export:
    raise ValueError("Baseline-Bonusauswertung passt nicht; zuerst Notebook 06a ausführen.")
baseline_metrics = pd.read_csv(TABLES / "task6_baseline_metrics.csv")
baseline_frequencies = pd.read_csv(TABLES / "task6_baseline_frequencies.csv")

run_config = make_run_config(
    expected_parameters | {
        "method": "cellcnn_prototype", "implementation_version": IMPLEMENTATION_VERSION,
        "model_format": "state_dict_and_scaler_v1", "top_fraction": TOP_FRACTION,
        "pooling": "top_fraction_mean_thresholded_responses", "l2_scope": "output_weight_only",
        "initialization_cells_per_donor": INITIALIZATION_CELLS_PER_DONOR,
        "initialization_seed_offset": INITIALIZATION_SEED_OFFSET,
        "initial_radius_quantile": 0.01, "eps": EPS, "device": str(DEVICE),
        "numpy": version("numpy"), "torch": version("torch"),
        "scikit_learn": version("scikit-learn"), "flowkit": version("flowkit"),
    }, [SPLITS_PATH, LABELS_PATH, MARKERS_PATH, *fcs_paths],
    PROJECT_ROOT / "notebooks/06c_cellcnn_mahalanobis_top1.ipynb", TRAINING_CODE_CELLS,
)
frequency_rows, prediction_parts, selection_parts, timing_rows = [], [], [], []
for split_id in active_split_ids:
    checkpoint_path = TABLES / f"task6_mahalanobis_top1_split_{split_id}.pt"
    if checkpoint_path.exists():
        checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
        if checkpoint["config"] != run_config or checkpoint["split_id"] != split_id:
            raise ValueError(f"Unpassender Bonus-Checkpoint: {checkpoint_path.name}; nicht überschrieben.")
        print(f"Mahalanobis-Top1-Split {split_id}: passenden Checkpoint geladen.", flush=True)
    else:
        if not run_training:
            raise FileNotFoundError(f"Fehlender Bonus-Checkpoint: {checkpoint_path.name}")
        started = time.monotonic()
        predictions, selection, selected = train_outer_split(split_id)
        checkpoint = {
            "config": run_config, "split_id": split_id,
            "state_dict": {name: value.detach().cpu() for name, value in selected["model"].state_dict().items()},
            "scaler_mean": selected["scaler"].mean_.tolist(), "scaler_scale": selected["scaler"].scale_.tolist(),
            "filter_count": int(selected["filter_count"]), "inner_fold": int(selected["inner_fold"]),
            "predictions": predictions.to_dict("records"), "selection": selection.to_dict("records"),
            "training_seconds": time.monotonic() - started,
        }
        temporary = checkpoint_path.with_suffix(".tmp")
        torch.save(checkpoint, temporary)
        temporary.replace(checkpoint_path)
        del selected
        print(f"Mahalanobis-Top1-Split {split_id}: {checkpoint['training_seconds'] / 60:.1f} Minuten; gespeichert.", flush=True)

    predictions = pd.DataFrame(checkpoint["predictions"])
    selection = pd.DataFrame(checkpoint["selection"])
    validate_prediction_splits(predictions, donor_splits)
    assert set(predictions.split_id) == {split_id} and predictions.score.between(0, 1).all()
    assert len(selection) == 9 and selection.selected.sum() == 1
    best = selection.sort_values(
        ["validation_accuracy", "validation_roc_auc", "best_validation_loss", "filter_count", "inner_fold"],
        ascending=[False, False, True, True, True]).iloc[0]
    assert bool(best.selected) and best.inner_fold == checkpoint["inner_fold"] and best.filter_count == checkpoint["filter_count"]
    scaler = restore_scaler(pd.DataFrame({"scaler_mean": checkpoint["scaler_mean"], "scaler_scale": checkpoint["scaler_scale"]}))
    model = PrototypeCellCNN(len(markers), checkpoint["filter_count"]).to(DEVICE)
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()
    assert all(torch.isfinite(p).all() for p in model.parameters())
    assert torch.all(model.marker_weights() > 0) and torch.all(model.radii() > 0)
    torch.testing.assert_close(model.marker_weights().mean(-1), torch.ones(checkpoint["filter_count"], device=DEVICE))
    filter_id = strongest_positive_filter(model)
    split = donor_splits.loc[donor_splits.split_id.eq(split_id)]
    test_ids = sorted(split.loc[split.outer_partition.eq("test"), "donor_id"])
    with torch.no_grad():
        for i, donor in enumerate(test_ids):
            score = predict_donor(model, scaler, donor, int(split.split_seed.iloc[0]) + 900000 + i)
            expected_score = predictions.loc[predictions.donor_id.eq(donor), "score"].item()
            assert abs(score - expected_score) <= 1e-6
            indices = evaluation_indices[donor]
            frequency = np.nan
            if filter_id is not None:
                values = torch.from_numpy(scaler.transform(data_by_donor[donor][indices])).to(DEVICE)
                frequency = float((model.distances(values)[:, filter_id] < model.radii()[filter_id]).float().mean())
            frequency_rows.append({"split_id": split_id, "donor_id": donor, "y_true": label_by_donor[donor],
                                   "filter_id": filter_id, "frequency": frequency, "n_cells": len(indices)})
    prediction_parts.append(predictions)
    selection_parts.append(selection)
    timing_rows.append({"split_id": split_id, "training_seconds": checkpoint["training_seconds"]})
    modified_predictions = pd.concat(prediction_parts, ignore_index=True)
    modified_frequencies = pd.DataFrame(frequency_rows)
    modified_predictions.to_csv(TABLES / "task6_mahalanobis_top1_predictions.csv", index=False)
    modified_frequencies.to_csv(TABLES / "task6_mahalanobis_top1_frequencies.csv", index=False)
    pd.concat(selection_parts, ignore_index=True).to_csv(TABLES / "task6_mahalanobis_top1_selection.csv", index=False)
    pd.DataFrame(timing_rows).to_csv(TABLES / "task6_mahalanobis_top1_timing.csv", index=False)

modified_metrics = pd.DataFrame([summarize_split(modified_predictions, modified_frequencies, s) for s in active_split_ids])
modified_metrics.to_csv(TABLES / "task6_mahalanobis_top1_metrics.csv", index=False)


Mahalanobis-Top1-Split 0: passenden Checkpoint geladen.


Mahalanobis-Top1-Split 1: passenden Checkpoint geladen.


Mahalanobis-Top1-Split 2: passenden Checkpoint geladen.


## Gepaarter Vergleich

Alle AUCs werden auf den sechs äußeren Testspendern eines Splits berechnet. Wir vergleichen die Methoden pro Split und fassen diese Werte deskriptiv zusammen; mehrfach getestete Spender werden nicht als neue unabhängige Beobachtungen behandelt. Bei fehlenden Phänotypmetriken berichten wir die Zahl bestimmbarer Splits. Unterschiedliche Membership-Regeln schränken den direkten Vergleich der Populationen ein.

Die Network-AUC-Tabelle enthält zusätzlich die vorhandene Mean-Variante aus 06b. Vor der Übernahme werden deren Full-Konfiguration, Eingabedaten und Trainingscode geprüft. So ist der Vergleich Top1 gegen Mean gepaart auf denselben drei Splits. Die Phänotypauswertung der neuen Variante wird weiterhin mit der Baseline verglichen.

In [11]:
# Der Mean-Vergleich verwendet die gespeicherten Full-Modelle aus 06b.
mean_config = make_run_config(
    run_config["parameters"] | {"implementation_version": "diagonal_prototype_relu_mean_v1",
        "top_fraction": None, "pooling": "mean_all_thresholded_responses"},
    [SPLITS_PATH, LABELS_PATH, MARKERS_PATH, *fcs_paths],
    PROJECT_ROOT / "notebooks/06b_cellcnn_mahalanobis_relu_threshold.ipynb", TRAINING_CODE_CELLS,
)
mean_prediction_parts = []
for split_id in active_split_ids:
    saved_mean = torch.load(TABLES / f"task6_modified_split_{split_id}.pt", map_location="cpu", weights_only=True)
    if saved_mean["config"] != mean_config or saved_mean["split_id"] != split_id:
        raise ValueError(f"Mean-Referenz für Split {split_id} passt nicht zu 06b und den Full-Einstellungen.")
    mean_prediction_parts.append(pd.DataFrame(saved_mean["predictions"]))
mean_predictions = pd.concat(mean_prediction_parts, ignore_index=True)
validate_prediction_splits(mean_predictions, donor_splits)
mean_auc = pd.DataFrame([{"split_id": s, "network_auc_mean": roc_auc_score(g.y_true, g.score)}
                         for s, g in mean_predictions.groupby("split_id")])

comparison = baseline_metrics.merge(modified_metrics, on="split_id", suffixes=("_baseline", "_modified"), validate="one_to_one")
assert set(comparison.split_id) == set(active_split_ids)
comparison = comparison.merge(mean_auc, on="split_id", validate="one_to_one")
comparison["network_auc_delta"] = comparison.network_auc_modified - comparison.network_auc_baseline
comparison["network_auc_delta_vs_mean"] = comparison.network_auc_modified - comparison.network_auc_mean
# Der Vergleich enthält exakt dieselben Spender und Zellzahlen je Split.
keys = ["split_id", "donor_id", "y_true", "n_cells"]
paired = baseline_frequencies.merge(modified_frequencies, on=keys, validate="one_to_one")
assert len(paired) == len(modified_frequencies) == 6 * len(active_split_ids)
display(comparison[["split_id", "network_auc_baseline", "network_auc_mean", "network_auc_modified", "network_auc_delta", "network_auc_delta_vs_mean"]].rename(columns={
    "network_auc_baseline": "Baseline-AUC", "network_auc_mean": "Mahalanobis Mean-AUC", "network_auc_modified": "Mahalanobis Top1-AUC",
    "network_auc_delta": "Top1 minus Baseline", "network_auc_delta_vs_mean": "Top1 minus Mean"}))
summary = pd.DataFrame({
    "Metrik": ["Mittlere Network-ROC-AUC", "Mediane Network-ROC-AUC", "Mittlere Häufigkeits-ROC-AUC",
               "Mittlere Häufigkeitsdifferenz CMV+ minus CMV−"],
    "Baseline": [comparison.network_auc_baseline.mean(), comparison.network_auc_baseline.median(),
                 comparison.frequency_auc_baseline.mean(), comparison.frequency_effect_baseline.mean()],
    "Mahalanobis Top1": [comparison.network_auc_modified.mean(), comparison.network_auc_modified.median(),
                 comparison.frequency_auc_modified.mean(), comparison.frequency_effect_modified.mean()],
})
display(summary)
print(f"Mittlere AUC-Differenz: {comparison.network_auc_delta.mean():.4f}; Median: {comparison.network_auc_delta.median():.4f}")
print(f"Mittlere Network-AUC der Mean-Variante: {comparison.network_auc_mean.mean():.4f}; "
      f"mittlere Top1-minus-Mean-Differenz: {comparison.network_auc_delta_vs_mean.mean():.4f}; "
      f"Median: {comparison.network_auc_delta_vs_mean.median():.4f}")
print(f"Ausgewertete Full-Splits: {active_split_ids}; vorgesehen: {BONUS_SPLIT_IDS}")
print("Bestimmbare Phänotypmetriken:",
      {name: int(comparison[f"frequency_auc_{name}"].notna().sum()) for name in ["baseline", "modified"]})
display(comparison[["split_id", "frequency_auc_baseline", "frequency_auc_modified",
                    "frequency_effect_baseline", "frequency_effect_modified", "phenotype_status_baseline", "phenotype_status_modified"]])
comparison.to_csv(TABLES / "task6_mahalanobis_top1_paired_comparison.csv", index=False)
summary.to_csv(TABLES / "task6_mahalanobis_top1_comparison_summary.csv", index=False)


,split_id,Baseline-AUC,Mahalanobis Mean-AUC,Mahalanobis Top1-AUC,Top1 minus Baseline,Top1 minus Mean
0,0,1.0,0.375,0.375,-0.625,0.0
1,1,1.0,1.000,1.000,0.000,0.0
2,2,1.0,0.250,0.750,-0.250,0.5


,Metrik,Baseline,Mahalanobis Top1
0,Mittlere Network-ROC-AUC,1.000000,0.708333
1,Mediane Network-ROC-AUC,1.000000,0.750000
2,Mittlere Häufigkeits-ROC-AUC,1.000000,0.604167
3,Mittlere Häufigkeitsdifferenz CMV+ minus CMV−,0.022537,0.006808


Mittlere AUC-Differenz: -0.2917; Median: -0.2500
Mittlere Network-AUC der Mean-Variante: 0.5417; mittlere Top1-minus-Mean-Differenz: 0.1667; Median: 0.0000
Ausgewertete Full-Splits: [0, 1, 2]; vorgesehen: [0, 1, 2]
Bestimmbare Phänotypmetriken: {'baseline': 3, 'modified': 3}


,split_id,frequency_auc_baseline,frequency_auc_modified,frequency_effect_baseline,frequency_effect_modified,phenotype_status_baseline,phenotype_status_modified
0,0,1.0,0.3125,0.007350,-0.001275,berechnet,berechnet
1,1,1.0,0.6250,0.033825,0.010263,berechnet,berechnet
2,2,1.0,0.8750,0.026437,0.011438,berechnet,berechnet


Eine verbesserte Klassifikation ist eine zu prüfende Hypothese. Die wenigen Splits begründen weder Signifikanz noch allgemeine Überlegenheit. Die Frequenzmetriken sind ergänzende CMV-Assoziationen und keine Zelltypvalidierung. Auf Stabilitätsanalyse, neue Projektionen und zusätzliche Architekturvarianten wird verzichtet. Da die Pooling-Hypothese nach Sichtung der bisherigen Ergebnisse entstand und dieselben Spender erneut verwendet werden, bleibt dieser Vergleich explorativ; er ist keine unabhängige Bestätigung.